# RankGenerator
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/parallel_state.py#L447

- 这里实例化了一个 RankGenerator 对象。你可以把它想象成一个“排兵布阵的指挥官”，它根据你传入的各个并行度（TP、EP、DP、PP等），负责计算出哪些 GPU 应该被分到一个组里进行通信。
   

In [ ]:
    expert_decoder_rank_generator = RankGenerator(
        tp=expert_tensor_parallel_size,
        ep=expert_model_parallel_size,
        dp=expert_data_parallel_size,
        pp=pipeline_model_parallel_size,
        cp=1, # 上下文并行（Context Parallelism）设为 1，表示当前 MoE 层暂时不启用上下文并行。
        order=order, # 并行策略的嵌套顺序（例如 'tp-ep-dp-pp'），决定了 GPU 分组的层级关系。
        rank_offset=rank_offset,
    )


In [ ]:
class RankGenerator(object):
    """A class for generating rank groups for different modes of parallelism."""

    def __init__(
        self, tp: int, ep: int, dp: int, pp: int, cp: int, order: str, rank_offset: int = 0
    ) -> None:
        '''
        这里使用了 assert 进行防御性编程。它强制要求 EP（专家并行）和 CP（上下文并行）不能同时大于 1。
        '''
        assert (
            ep == 1 or cp == 1
        ), "Both EP and CP > 1 in not allow in one rank generator. \
            CP is only included in default RankGenerator, and EP only in expert RankGenerator."

        self.tp = tp
        self.ep = ep
        self.dp = dp
        self.pp = pp
        self.cp = cp
        self.rank_offset = rank_offset # 进程排名的偏移量，默认为 0。
        self.world_size = tp * dp * pp * cp * ep

        self.name_to_size = {
            "tp": self.tp,
            "pp": self.pp,
            "dp": self.dp,
            "ep": self.ep,
            "cp": self.cp,
        }
        self.order = order
        order = order.lower()

        for name in self.name_to_size.keys():
            # 如果某个并行度大于1，但在 order 字符串里没提到，就报错
            if name not in order and self.name_to_size[name] != 1:
                raise RuntimeError(
                    f"The size of ({name}) is ({self.name_to_size[name]}), but you haven't"
                    f"specified the order ({self.order})."
                )
            elif name not in order:
                '''
                如果某个并行度等于1（即没开启该并行），且在 order 里没提到，就自动补上 "-name"
                
                如果某个维度没开启（等于1），即使用户没写在 order 里，程序也会自动把它追加到字符串末尾。
                这样做是为了保证后续遍历 order 时，所有维度都不会被遗漏。
                '''
                order = order + "-" + name

        self.order = order # 例如 "tp-dp-pp-ep-cp"
        self.ordered_size = []

        for token in order.split("-"):
            self.ordered_size.append(self.name_to_size[token])

    def get_mask(self, order: str, token: str):
        """Create a mask for the specified tokens based on the given order.

        Args:
            order (str): The order of parallelism types (e.g., 'tp-dp-pp').
            token (str): The specific parallelism types to include in the mask,
                         separated by hyphens (e.g., 'tp-dp').
        """
        ordered_token = order.split("-")
        token_list = token.split("-")
        
        # 初始 mask 为 [False, False, False]
        mask = [False] * len(ordered_token)
        for t in token_list:
            mask[ordered_token.index(t)] = True
        return mask

    def get_ranks(self, token):
        """Get rank group by input token.

        Args:
            token (str):
                Specify the ranks type that want to get. If we want
                to obtain multiple parallel types, we can use a hyphen
                '-' to separate them. For example, if we want to obtain
                the TP_DP group, the token should be 'tp-dp'.
        """
        mask = self.get_mask(self.order, token)
        '''
        计算正交分组 :
        '''
        ranks = generate_masked_orthogonal_rank_groups(self.world_size, self.ordered_size, mask)
        
        '''
        应用排名偏移量:
        
        这是一个边界处理逻辑。在某些复杂的分布式场景下（比如跨多个节点或与其他框架混合训练时），
        当前这批进程的编号可能不是从 0 开始的，而是有一个基础的偏移量。
        
        如果初始化时设置了 self.rank_offset > 0，这里就会遍历刚刚生成的所有分组，
        把组内每一个 rank 的编号都加上这个偏移量，从而得到最终在全局环境下的真实 rank 编号。
        '''
        if self.rank_offset > 0:
            for rank_group in ranks:
                for i in range(len(rank_group)):
                    rank_group[i] += self.rank_offset
        return ranks
        '''
        将最终计算好的、包含真实 rank 编号的二维列表返回给调用者。调用者拿到这个列表后，
        就可以使用 torch.distributed.new_group(ranks) 来正式建立底层的通信组。
        '''


# get_ranks(self, token):
- 主要作用是：根据用户指定的并行策略类型（Token），动态生成对应的进程组（Rank Groups）列表。
- 简单来说，在分布式训练中，我们需要把成千上万个 GPU（每个 GPU 对应一个 rank）按照不同的并行策略（如数据并行 DP、张量并行 TP 等）划分成一个个小团体。这个方法就是用来生成分组名单的。

<font color='red'>token (str)</font>：一个字符串参数，用来告诉程序你想要获取哪种并行策略的分组。
  - 如果你想获取单一策略，比如数据并行，token 就是 "dp"。
  - 如果你想获取组合策略，比如张量并行加数据并行，可以用连字符连接，token 就是 "tp-dp

# get_mask(self.order, token)

- 它的作用是将你传入的 token（比如 "tp-dp"）和初始化时设定的全局 order（比如 "tp-dp-pp"）进行比对。
- 最终生成一个布尔值列表（即掩码 mask）。例如，如果 order 是 ["tp", "dp", "pp"]，而你传入的 token 是 "tp-dp"，那么生成的 mask 可能就是 [True, True, False]。
- <font color='red'>这个掩码告诉后续的计算逻辑：“我需要把 TP 和 DP 这两个维度组合在一起，而忽略 PP”。</font>

# generate_masked_orthogonal_rank_groups

它接收三个关键参数：
- self.world_size：总的 GPU 进程数量。
- self.ordered_size：你在 __init__ 中生成的按顺序排列的并行度列表（例如 [2, 4, 2] 代表 TP=2, DP=4, PP=2）。
- mask：上一步生成的掩码，指明要组合哪些维度。

该函数会根据这些参数，通过数学计算（通常涉及多维坐标到一维 rank 的映射），返回一个二维列表 ranks。比如 [[0, 1], [2, 3], [4, 5]]，其中每一个小列表 [0, 1] 就代表一个具体的通信进程组。




In [ ]:
r"""根据并行维度的大小和掩码（mask）生成正交的并行进程组。

    参数:
        world_size (int): 全局总的进程数量（world size）。

        parallel_size (List[int]):
            各个正交并行类型的并行度大小。例如，如果
            张量并行度(tp)=2，流水线并行度(pp)=3，数据并行度(dp)=4，
            且并行映射顺序为 tp-pp-dp，那么 parallel_size = [2, 3, 4]。

        mask (List[bool]):
            掩码用于控制生成的进程组代表哪些并行方法。如果 mask[i] 为
            True，则表示生成的组包含第 i 种并行方法。例如，
            如果 parallel_size = [tp_size, pp_size, dp_size]，且 mask = [True, False, True]，那么
            生成的组就是 `tp-dp` 组；如果 mask = [False, True, False]，那么生成的
            组就是 `pp` 组。

    算法原理:
        对于正交并行（如 tp/dp/pp/cp），全局进程编号（global_rank）和
        局部进程编号（local_rank）满足以下等式：
            global_rank = tp_rank + dp_rank * tp_size + pp_rank * tp_size * dp_size (1)
                其中 tp_rank 取值范围为 [0, tp_size)
                     dp_rank 取值范围为 [0, dp_size)
                     pp_rank 取值范围为 [0, pp_size)

        如果我们想获取 `dp_group`（即 tp_size * pp_size 个组，每组包含 dp_size 个进程。
        例如，如果 GPU 总数为 8，顺序是 'tp-pp-dp'，大小是 '2-2-2'，那么
        这里的 dp_group 就是 [[0, 4], [1, 5], [2, 6], [3, 7]]。）
        此时，tp_rank 和 pp_rank 将被组合起来形成 `dp_group_index`（数据并行组的索引）。
            dp_group_index = tp_rank + pp_rank * tp_size (2)

        因此，给定满足等式 (2) 的 tp_rank 和 pp_rank，且 dp_rank 在
        range(0, dp_size) 范围内时，dp_group[dp_group_index] 中的进程编号满足
        等式 (1)。

        本函数就是为了解决上述数学问题。

    举例说明:
        如果 parallel_size = [tp_size, dp_size, pp_size] = [2, 3, 4]，
        且 mask = [False, True, False]。那么：
            dp_group_index(0) = tp_rank(0) + pp_rank(0) * 2
            dp_group_index(1) = tp_rank(1) + pp_rank(0) * 2
            ...
            dp_group_index(7) = tp_rank(1) + pp_rank(3) * 2

            dp_group[0] = 0 + range(0, 3) * 2 + 0 = [0, 2, 4]
            dp_group[1] = 1 + range(0, 3) * 2 + 0 = [1, 3, 5]
            ...
            dp_group[7] = 1 + range(0, 3) * 2 + 3 * 2 * 3 = [19, 21, 23]
    """

In [ ]:
def generate_masked_orthogonal_rank_groups(
    world_size: int, parallel_size: List[int], mask: List[bool]
) -> List[List[int]]:
    r"""Generate orthogonal parallel groups based on the parallel size and mask.
    
    Arguments:
        world_size (int): world size

        parallel_size (List[int]):
            The parallel size of each orthogonal parallel type. For example, if
            tensor_parallel_size = 2, pipeline_model_parallel_group = 3, data_parallel_size = 4,
            and the parallel mapping order is tp-pp-dp, then the parallel_size = [2, 3, 4].

        mask (List[bool]):
            The mask controls which parallel methods the generated groups represent. If mask[i] is
            True, it means the generated group contains the i-th parallelism method. For example,
            if parallel_size = [tp_size, pp_size, dp_size], and mask = [True, False , True], then
            the generated group is the `tp-dp` group, if the mask = [False, True, False], then the
            generated group is the `pp` group.

    Algorithm: 分布式训练中进程分组与编号映射
    
    每个进程有「全局编号（global_rank）」和「局部编号（local_rank，即各并行维度内的排名）」，
    需通过数学关系关联。
    
    - tp_size：张量并行的进程数（一个tp组内的进程数量）。
    - dp_size：数据并行的进程数（一个dp组内的进程数量）。
    - pp_size：流水线并行的进程数（一个pp组内的进程数量）。
    
    - tp_rank：进程在张量并行组内的局部排名（范围 [0, tp_size)）。
    - dp_rank：进程在数据并行组内的局部排名（范围 [0, dp_size)）。
    - pp_rank：进程在流水线并行组内的局部排名（范围 [0, pp_size)）。
    - global_rank：进程的全局唯一编号（所有进程中不重复）。
    
    - dp_group_index：数据并行组的索引（用于标记“哪些进程属于同一个dp组”）。
    
    逻辑：- 先按 tp 维度排列（每个tp组占 tp_size 个位置），
         - 再按 dp 维度扩展（每个dp组包含 tp_size 个tp组），
         - 最后按 pp 维度扩展（每个pp组包含 tp_size×dp_size 个进程）。
         
        For orthogonal parallelism, such as tp/dp/pp/cp, the global_rank and
        local_rank satisfy the following equation:
            global_rank = tp_rank + dp_rank * tp_size + pp_rank * tp_size * dp_size (1)
                tp_rank \in [0, tp_size)
                dp_rank \in [0, dp_size)
                pp_rank \in [0, pp_size)

        If we want to get the `dp_group` (tp_size * pp_size groups of dp_size ranks each.
        For example,  if the gpu size is 8 and order is 'tp-pp-dp', size is '2-2-2', and the
        dp_group here is [[0, 4], [1, 5], [2, 6], [3, 7]].)
        The tp_rank and pp_rank will be combined to form the `dp_group_index`.
        
        数据并行组的核心是“相同dp组内的进程共享同一份数据”，因此需将 tp_rank 和 pp_rank 组合为唯一的 dp_group_index：
        
        逻辑：tp_rank 是tp维度内的偏移，pp_rank × tp_size 是pp维度对tp的“块偏移”（每个pp块包含 tp_size 个tp进程），
        两者相加得到跨tp、pp维度的唯一标识，用于区分不同的dp组。
        
            dp_group_index = tp_rank + pp_rank * tp_size (2)

        So, Given that tp_rank and pp_rank satisfy equation (2), and dp_rank in
        range(0, dp_size), the ranks in dp_group[dp_group_index] satisfies the
        equation (1).

        This function solve this math problem.

    For example, if the parallel_size = [tp_size, dp_size, pp_size] = [2, 3, 4],
    and the mask = [False, True, False]. Then,
        dp_group_index(0) = tp_rank(0) + pp_rank(0) * 2
        dp_group_index(1) = tp_rank(1) + pp_rank(0) * 2
        ...
        dp_group_index(7) = tp_rank(1) + pp_rank(3) * 2

        dp_group[0] = 0 + range(0, 3) * 2 + 0 = [0, 2, 4]
        dp_group[1] = 1 + range(0, 3) * 2 + 0 = [1, 3, 5]
        ...
        dp_group[7] = 1 + range(0, 3) * 2 + 3 * 2 * 3 = [19, 21, 23]
    """

    def prefix_product(a: List[int], init=1) -> List[int]:
        r = [init]
        for v in a:
            init = init * v
            r.append(init)
        return r

    def inner_product(a: List[int], b: List[int]) -> int:
        return sum([x * y for x, y in zip(a, b)])

    def decompose(index, shape, stride=None):
        """
        This function solve the math problem below:
            There is an equation:
                index = sum(idx[i] * stride[i])
            And given the value of index, stride.
            Return the idx.
        This function will be used to get the pp/dp/pp_rank
        from group_index and rank_in_group.
        """
        if stride is None:
            stride = prefix_product(shape)
        idx = [(index // d) % s for s, d in zip(shape, stride)]
        # stride is a prefix_product result. And the value of stride[-1]
        # is not used.
        assert (
            sum([x * y for x, y in zip(idx, stride[:-1])]) == index
        ), "idx {} with shape {} mismatch the return idx {}".format(index, shape, idx)
        return idx

    masked_shape = [s for s, m in zip(parallel_size, mask) if m]
    unmasked_shape = [s for s, m in zip(parallel_size, mask) if not m]

    global_stride = prefix_product(parallel_size)
    masked_stride = [d for d, m in zip(global_stride, mask) if m]
    unmasked_stride = [d for d, m in zip(global_stride, mask) if not m]

    group_size = prefix_product(masked_shape)[-1]
    num_of_group = world_size // group_size

    ranks = []
    for group_index in range(num_of_group):
        # get indices from unmaksed for group_index.
        decomposed_group_idx = decompose(group_index, unmasked_shape)
        rank = []
        for rank_in_group in range(group_size):
            # get indices from masked for rank_in_group.
            decomposed_rank_idx = decompose(rank_in_group, masked_shape)
            rank.append(
                inner_product(decomposed_rank_idx, masked_stride)
                + inner_product(decomposed_group_idx, unmasked_stride)
            )
        ranks.append(rank)
    return ranks

